# Introduction
In this notebook, we will filter the data to adhere to whats important to the requirements of the project
**NYC311** has an *enormous* dataset, as such its important to keep ONLY features which are important 
when conducting exploratory data analysis and any other process within this regression project.

The filtered dataset will be available in this repository. Any user is free to use their own sourced
**NYC311** data with this notebook and watch the results turn the same.

As the csv files remain too large to upload, we will not include them in this repository.

# Setup

In [84]:
# setup parameters
# --------------------------------------------- #
# edit this to choose the year of data you want #
year = 2025
filepath = '../data/nyc311.csv'
# --------------------------------------------- #

In [85]:
# imports
import pandas as pd
import numpy as np

In [86]:
# grab dataset
df = pd.read_csv(filepath, index_col='Unique Key')

/var/folders/x4/j70qy1ns0gdg8282hfmvlbm80000gn/T/ipykernel_5058/3878635956.py:2: DtypeWarning: Columns (0: Incident Address) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath, index_col='Unique Key')


In [87]:
df.info(show_counts=True)

<class 'pandas.DataFrame'>
Index: 9386300 entries, 69841519 to 59899838
Data columns (total 23 columns):
 #   Column                                Non-Null Count    Dtype  
---  ------                                --------------    -----  
 0   Created Date                          9386300 non-null  str    
 1   Closed Date                           9124451 non-null  str    
 2   Agency                                9386300 non-null  str    
 3   Agency Name                           9386300 non-null  str    
 4   Problem (formerly Complaint Type)     9386300 non-null  str    
 5   Problem Detail (formerly Descriptor)  9146140 non-null  str    
 6   Additional Details                    3681379 non-null  str    
 7   Location Type                         8211512 non-null  str    
 8   Incident Zip                          9300914 non-null  object 
 9   Incident Address                      9043260 non-null  str    
 10  Street Name                           9042989 non-null  str   

In [88]:
df.head()

,Created Date,Closed Date,Agency,Agency Name,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Additional Details,Location Type,Incident Zip,Incident Address,...,Landmark,Status,Community Board,Council District,Police Precinct,BBL,Borough,Open Data Channel Type,Latitude,Longitude
Unique Key,,,,,,,,,,,,,,,,,,,,,
69841519,07/26/2026 02:05:49 AM,NaN,NYPD,New York City Police Department,Illegal Parking,Posted Parking Sign Violation,NaN,Street/Sidewalk,10031.0,571 RIVERSIDE DRIVE,...,RIVERSIDE DRIVE,In Progress,09 MANHATTAN,7.0,Precinct 30,1.020010e+09,MANHATTAN,ONLINE,40.820714,-73.958313
69840231,07/26/2026 02:05:47 AM,NaN,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,10466.0,4379 WHITE PLAINS ROAD,...,WHITE PLAINS ROAD,In Progress,12 BRONX,11.0,Precinct 47,2.050420e+09,BRONX,ONLINE,40.897833,-73.854802
69840191,07/26/2026 02:05:43 AM,NaN,NYPD,New York City Police Department,Noise - Commercial,Loud Music/Party,NaN,Store/Commercial,11368.0,37-30 103 STREET,...,103 STREET,In Progress,03 QUEENS,21.0,Precinct 115,4.017688e+09,QUEENS,MOBILE,40.752822,-73.864228
69841574,07/26/2026 02:05:40 AM,NaN,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,10472.0,1326 MANOR AVENUE,...,MANOR AVENUE,In Progress,09 BRONX,17.0,Precinct 43,2.038660e+09,BRONX,ONLINE,40.832249,-73.877155
69838845,07/26/2026 02:05:37 AM,NaN,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,11691.0,22-25 NEW HAVEN AVENUE,...,NEW HAVEN AVENUE,In Progress,14 QUEENS,31.0,Precinct 101,4.157620e+09,QUEENS,ONLINE,40.601082,-73.756295


# Column dropping
We are only going to keep a small subset of columns which will be directly useful in regression. The
columns which will be dropped, and their reasons are as follows:

| Column Name | Reason |
|---|---|
| Agency | `Agency Name` has the same information |
| Additional Details | Not always utilized (many null), would lead to a sparse input |
| Incident Zip | `Latitude` and `Longitude` will cover location |
| Status | A problem is closed if `Closed Date` is not null, therefore this is redundant |
| Location Type | Many nulls, `Latitude` and `Longitude` will cover location |
| Incident Address | `Latitude` and `Longitude` will cover location |
| Street Name | `Latitude` and `Longitude` will cover location |
| Landmark | `Latitude` and `Longitude` will cover location |
| City | `Latitude` and `Longitude` will cover location |
| Community Board | Equivalent to borough, which will be found with `Latitude` and `Longitude` |
| Council District | Location-based, `Latitude` and `Longitude` will cover location |
| Police Precinct | Location-based, `Latitude` and `Longitude` will cover location |
| BBL | Location-based, `Latitude` and `Longitude` will cover location |

---


In [89]:
cols = [
    'Created Date',
    'Closed Date',
    'Agency Name',
    'Problem (formerly Complaint Type)',
    'Problem Detail (formerly Descriptor)',
    'Borough',
    'Latitude',
    'Longitude',
    'Open Data Channel Type'
]

df = df[cols]

In other words, we are keeping the following:
| Column Name | Description |
|---|---|
| Unique Key | Identifier column |
| Created Date | Date incident created |
| Closed Date | Date incident closed |
| Agency Name | Name of agency which responded to incident |
| Problem (formerly Complaint Type) | Category of incident |
| Problem Detail (formerly Descriptor) | Category of incident, with detail. Although this has many nulls, we can combine with `Problem (formerly Complaint Type)` to create a more descriptive set of problems. |
| Borough | Location of incident |
| Open Data Channel Type | Method which **NYC311** was contacted |
| Latitude | Location of incident |
| Longitude | Location of incident |

In [90]:
# rename
df = df.rename(columns={
        'Created Date': 'created',
        'Closed Date': 'closed',
        'Agency Name': 'agency_name',
        'Problem (formerly Complaint Type)' : 'problem',
        'Problem Detail (formerly Descriptor)' : 'detail',
        'Borough' : 'borough',
        'Latitude' : 'lat',
        'Longitude' : 'long',
        'Open Data Channel Type': 'method'
    }
)
df.index.name = "id"

In [91]:
df.info(show_counts=True)

<class 'pandas.DataFrame'>
Index: 9386300 entries, 69841519 to 59899838
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   created      9386300 non-null  str    
 1   closed       9124451 non-null  str    
 2   agency_name  9386300 non-null  str    
 3   problem      9386300 non-null  str    
 4   detail       9146140 non-null  str    
 5   borough      9386300 non-null  str    
 6   lat          9224820 non-null  float64
 7   long         9224820 non-null  float64
 8   method       9386300 non-null  str    
dtypes: float64(2), str(7)
memory usage: 716.1 MB


# Calculate Resolution Time
This project is aimed at predicting resolution time, as such, we will create a column, `resolution_time`,
which is the difference between `created` and `closed`, in hours, to three decimal places.

In [92]:
# set time columns
df['created'] = pd.to_datetime(df['created'], format='%m/%d/%Y %I:%M:%S %p')
df['closed'] = pd.to_datetime(df['closed'], format='%m/%d/%Y %I:%M:%S %p')

In [93]:
# create resolution time
df['resolution_time'] = np.round((df['closed'] - df['created']).dt.total_seconds() / 3600, 3)

In [94]:
df.info(show_counts=True)

<class 'pandas.DataFrame'>
Index: 9386300 entries, 69841519 to 59899838
Data columns (total 10 columns):
 #   Column           Non-Null Count    Dtype         
---  ------           --------------    -----         
 0   created          9386300 non-null  datetime64[us]
 1   closed           9124451 non-null  datetime64[us]
 2   agency_name      9386300 non-null  str           
 3   problem          9386300 non-null  str           
 4   detail           9146140 non-null  str           
 5   borough          9386300 non-null  str           
 6   lat              9224820 non-null  float64       
 7   long             9224820 non-null  float64       
 8   method           9386300 non-null  str           
 9   resolution_time  9124451 non-null  float64       
dtypes: datetime64[us](2), float64(3), str(5)
memory usage: 787.7 MB


# Row Filtering
Here, we will filter our data with the following:

| Row | Action |
|---|---|
| `(all rows)` | Remove nulls, we will only keep nulls for the `detail` column |
| `created` | Keep only data within the specified `year` |
| `resolution_time` | Remove instances where `resolution_time` is negative (`closed` is before `created`) |

In [95]:
# created & status filter
df = df[df['created'].dt.year == year] # will also remove nulls

In [96]:
# drop nulls 
df = df.dropna(subset=['closed', 'agency_name', 'problem', 'borough', 'lat', 'long', 'method'])

In [97]:
# filter any error situations, where closed is **before** created
# although this is something that can be seen in datapoints where for example, somebody calls for 
# problem which was already fixed, this is not useful information when trying to predict resolution
# time, for a problem which would be presumed to be confirmed not closed.
df = df[df['resolution_time'] > 0.0]

In [98]:
df.info(show_counts=True)

<class 'pandas.DataFrame'>
Index: 3486292 entries, 67351762 to 63577994
Data columns (total 10 columns):
 #   Column           Non-Null Count    Dtype         
---  ------           --------------    -----         
 0   created          3486292 non-null  datetime64[us]
 1   closed           3486292 non-null  datetime64[us]
 2   agency_name      3486292 non-null  str           
 3   problem          3486292 non-null  str           
 4   detail           3409209 non-null  str           
 5   borough          3486292 non-null  str           
 6   lat              3486292 non-null  float64       
 7   long             3486292 non-null  float64       
 8   method           3486292 non-null  str           
 9   resolution_time  3486292 non-null  float64       
dtypes: datetime64[us](2), float64(3), str(5)
memory usage: 292.6 MB


# Download Dataset
We have completed filtering our dataset. We can now download this data for future use with this
project.

In [99]:
with open(f'../data/nyc311_{year}.csv', 'w') as f:
    df.to_csv(f)